# SessionSmith チュートリアル

このノートブックでは、SessionSmithの使い方を段階的に学びます。

## 目次

1. [基本的な使い方](#1-基本的な使い方)
2. [圧縮サポート](#2-圧縮サポート)
3. [セッション情報表示](#3-セッション情報表示)
4. [セレクティブロード](#4-セレクティブロード)
5. [セッション比較](#5-セッション比較)
6. [SessionManagerクラス](#6-sessionmanagerクラス)
7. [自動バックアップ](#7-自動バックアップ)
8. [カスタムシリアライザー](#8-カスタムシリアライザー)
9. [実践例](#9-実践例)



## 1. 基本的な使い方

まず、SessionSmithをインポートして、基本的な保存・復元の使い方を学びましょう。

**注意**: SessionSmithはJupyter Notebook環境で使用する場合、`_ih`, `_oh`, `In`, `Out`, `_i1`, `_i2`などの内部変数を自動的に除外します。これにより、不要な変数が保存されることを防ぎます。内部変数も保存したい場合は`exclude_jupyter=False`を指定してください。

チュートリアルで作成するファイルは`temp/`ディレクトリに保存されます。


In [1]:
# SessionSmithをインポート
from SessionSmith import save_session, load_session
import os

# tempディレクトリを作成（存在しない場合）
os.makedirs("temp", exist_ok=True)
print("✓ temp/ディレクトリを準備しました")

# いくつかの変数を作成
import numpy as np
import pandas as pd

# データを作成
data = {"name": "SessionSmith", "version": "0.1.1"}
numbers = [1, 2, 3, 4, 5]
array = np.array([1, 2, 3, 4, 5])
df = pd.DataFrame({"A": [1, 2, 3], "B": [4, 5, 6]})

print("変数を作成しました:")
print(f"  data: {data}")
print(f"  numbers: {numbers}")
print(f"  array: {array}")
print(f"  df shape: {df.shape}")


✓ temp/ディレクトリを準備しました
変数を作成しました:
  data: {'name': 'SessionSmith', 'version': '0.1.1'}
  numbers: [1, 2, 3, 4, 5]
  array: [1 2 3 4 5]
  df shape: (3, 2)


In [2]:
# セッションを保存
save_session("temp/my_session.pkl", verbose=True)
print("\n✓ セッションを保存しました")

Saved variable: data (dict)
Saved variable: numbers (list)
Saved variable: array (ndarray)
Saved variable: df (DataFrame)
Session saved to temp/my_session.pkl (859 bytes)
Warnings: 3 variables could not be saved

✓ セッションを保存しました


In [3]:
# 変数を削除してから復元をテスト
del data, numbers, array, df

# セッションを復元
load_session("temp/my_session.pkl", verbose=True)

# 復元された変数を確認
print("\n復元された変数:")
print(f"  data: {data}")
print(f"  numbers: {numbers}")
print(f"  array: {array}")
print(f"  df shape: {df.shape}")


Loaded variable: data (dict)
Loaded variable: numbers (list)
Loaded variable: array (ndarray)
Loaded variable: df (DataFrame)
Loaded 4 variables

復元された変数:
  data: {'name': 'SessionSmith', 'version': '0.1.1'}
  numbers: [1, 2, 3, 4, 5]
  array: [1 2 3 4 5]
  df shape: (3, 2)


## 2. 圧縮サポート

大きなセッションを圧縮して保存することで、ディスク容量を節約できます。


In [4]:
# 大きなデータを作成
large_data = np.random.rand(1000, 1000)

# 通常の保存
save_session("temp/session_normal.pkl")
size_normal = os.path.getsize("temp/session_normal.pkl")

# gzip圧縮で保存
save_session("temp/session_gzip.pkl", compress=True)
size_gzip = os.path.getsize("temp/session_gzip.pkl")

# bzip2圧縮で保存
save_session("temp/session_bz2.pkl", compress="bz2")
size_bz2 = os.path.getsize("temp/session_bz2.pkl")

print(f"通常: {size_normal:,} bytes")
print(f"gzip: {size_gzip:,} bytes ({size_gzip/size_normal*100:.1f}%)")
print(f"bz2:  {size_bz2:,} bytes ({size_bz2/size_normal*100:.1f}%)")


通常: 8,000,962 bytes
gzip: 7,544,782 bytes (94.3%)
bz2:  7,637,708 bytes (95.5%)


In [5]:
# 圧縮されたファイルも自動的に検出してロードできます
del large_data
load_session("temp/session_gzip.pkl")
print(f"✓ gzip圧縮ファイルから復元: large_data shape = {large_data.shape}")


✓ gzip圧縮ファイルから復元: large_data shape = (1000, 1000)


## 3. セッション情報表示

保存されたセッションファイルの詳細情報を確認できます。


In [6]:
from SessionSmith import get_session_info, print_session_info, list_session_variables

# セッション情報を取得
info = get_session_info("temp/my_session.pkl")
print("セッション情報:")
print(f"  ファイルサイズ: {info['file_size']:,} bytes")
print(f"  変数の数: {info['variable_count']}")
print(f"  総データサイズ: {info['total_data_size']:,} bytes")
print(f"  圧縮形式: {info['compression'] or 'なし'}")


セッション情報:
  ファイルサイズ: 859 bytes
  変数の数: 4
  総データサイズ: 977 bytes
  圧縮形式: なし


In [7]:
# 整形して表示
print_session_info("temp/my_session.pkl")


Session File: temp/my_session.pkl
File Size: 859 bytes
Modified: 2025-12-20T16:04:13.795044
Variables: 4
Total Data Size: 977 bytes

Variables:
  array                (ndarray        ) - 188 bytes
  data                 (dict           ) - 56 bytes
  df                   (DataFrame      ) - 707 bytes
  numbers              (list           ) - 26 bytes


In [8]:
# 変数名のリストを取得
variables = list_session_variables("temp/my_session.pkl")
print(f"保存されている変数: {variables}")


保存されている変数: ['array', 'data', 'df', 'numbers']


## 4. セレクティブロード

特定の変数のみをロードしたり、特定の変数を除外してロードできます。


In [9]:
# 新しい変数空間を作成
import sys
test_globals = {}

# 特定の変数のみをロード
load_session("temp/my_session.pkl", globals_dict=test_globals, include=["data", "numbers"])
print("ロードされた変数:", list(test_globals.keys()))


ロードされた変数: ['data', 'numbers']


In [10]:
# 特定の変数を除外してロード
test_globals2 = {}
load_session("temp/my_session.pkl", globals_dict=test_globals2, exclude=["df"])
print("ロードされた変数（dfを除外）:", list(test_globals2.keys()))


ロードされた変数（dfを除外）: ['data', 'numbers', 'array']


## 5. セッション比較

2つのセッションファイルを比較して、違いを確認できます。


In [11]:
# 最初のセッションを作成
data1 = {"version": 1, "count": 10}
save_session("temp/session1.pkl", exclude=["data", "numbers", "array", "df", "large_data"])

# 2番目のセッションを作成（変数を追加・変更）
data2 = {"version": 2, "count": 20}
new_var = "新しい変数"
save_session("temp/session2.pkl", exclude=["data", "numbers", "array", "df", "large_data"])


In [12]:
from SessionSmith import compare_sessions, print_comparison

# セッションを比較
result = compare_sessions("temp/session1.pkl", "temp/session2.pkl", detailed=True)
print("比較結果:")
print(f"  共通変数: {len(result['common_variables'])}")
print(f"  追加された変数: {result['added_variables']}")
print(f"  削除された変数: {result['removed_variables']}")
if 'changed_variables' in result:
    print(f"  変更された変数: {result['changed_variables']}")


比較結果:
  共通変数: 8
  追加された変数: ['data2', 'new_var']
  削除された変数: []
  変更された変数: []


In [13]:
# 整形して表示
print_comparison("temp/session1.pkl", "temp/session2.pkl", detailed=True)


Comparison: temp/session1.pkl vs temp/session2.pkl

Common variables (8):
  data1
  info
  size_bz2
  size_gzip
  size_normal
  test_globals
  test_globals2
  variables

Added variables (2):
  + data2
  + new_var


## 6. SessionManagerクラス

SessionManagerクラスを使うと、より便利にセッションを管理できます。


In [14]:
from SessionSmith import SessionManager

# マネージャーを作成
manager = SessionManager()

# 現在の変数を確認（Jupyter内部変数や組み込み関数を除外）
variables = manager.list_variables()
print(f"現在の変数 ({len(variables)}個):")
for var in variables[:10]:  # 最初の10個を表示
    print(f"  - {var}")
if len(variables) > 10:
    print(f"  ... 他 {len(variables) - 10}個の変数")


現在の変数 (30個):
  - SessionManager
  - array
  - compare_sessions
  - data
  - data1
  - data2
  - df
  - get_ipython
  - get_session_info
  - info
  ... 他 20個の変数


In [15]:
# マネージャーを使って保存
manager.save("temp/managed_session.pkl", metadata=True, verbose=True)


Saved variable: data (dict)
Saved variable: numbers (list)
Saved variable: array (ndarray)
Saved variable: df (DataFrame)
Saved variable: size_normal (int)
Saved variable: size_gzip (int)
Saved variable: size_bz2 (int)
Saved variable: large_data (ndarray)
Saved variable: info (dict)
Saved variable: variables (list)
Saved variable: test_globals (dict)
Saved variable: test_globals2 (dict)
Saved variable: data1 (dict)
Saved variable: data2 (dict)
Saved variable: new_var (str)
Saved variable: result (dict)
Saved variable: var (str)
Session saved to temp/managed_session.pkl (8002285 bytes)
Warnings: 4 variables could not be saved


In [16]:
# マネージャーを使ってロード
new_vars = manager.load("temp/managed_session.pkl", verbose=True)
print(f"\n✓ {len(new_vars)}個の変数がロードされました")


Session metadata: {'version': '0.1.1', 'saved_at': '2025-12-20T16:04:14.529714', 'variable_count': 17}
Loaded variable: data (dict)
Loaded variable: numbers (list)
Loaded variable: array (ndarray)
Loaded variable: df (DataFrame)
Loaded variable: size_normal (int)
Loaded variable: size_gzip (int)
Loaded variable: size_bz2 (int)
Loaded variable: large_data (ndarray)
Loaded variable: info (dict)
Loaded variable: variables (list)
Loaded variable: test_globals (dict)
Loaded variable: test_globals2 (dict)
Loaded variable: data1 (dict)
Loaded variable: data2 (dict)
Loaded variable: new_var (str)
Loaded variable: result (dict)
Loaded variable: var (str)
Loaded 17 variables

✓ 17個の変数がロードされました


## 7. 自動バックアップ

長時間実行する処理では、自動バックアップが便利です。


In [17]:
# 自動バックアップを開始（10秒ごと - デモ用）
# 実際の使用では、300秒（5分）などに設定することをお勧めします
manager.auto_save(interval=10, file_path="temp/autosave.pkl", metadata=True)
print("✓ 自動バックアップを開始しました（10秒ごと）")
print(f"  実行中: {manager.is_auto_save_running()}")


✓ 自動バックアップを開始しました（10秒ごと）
  実行中: True


In [18]:
import time

# 少し待機（実際の処理をシミュレート）
print("処理を実行中...")
for i in range(3):
    time.sleep(2)
    print(f"  ステップ {i+1}/3 完了")
    
print("\n自動バックアップが動作していることを確認:")
print(f"  実行中: {manager.is_auto_save_running()}")


処理を実行中...
  ステップ 1/3 完了
  ステップ 2/3 完了
  ステップ 3/3 完了

自動バックアップが動作していることを確認:
  実行中: True


In [19]:
# 自動バックアップを停止
manager.stop_auto_save()
print("✓ 自動バックアップを停止しました")
print(f"  実行中: {manager.is_auto_save_running()}")

# バックアップファイルが作成されているか確認
if os.path.exists("temp/autosave.pkl"):
    size = os.path.getsize("temp/autosave.pkl")
    print(f"  バックアップファイルサイズ: {size:,} bytes")


✓ 自動バックアップを停止しました
  実行中: False
  バックアップファイルサイズ: 8,002,552 bytes


## 8. カスタムシリアライザー

特定の型に対してカスタムシリアライゼーションを定義できます。


In [20]:
from SessionSmith import CustomSerializer

# カスタムクラスを定義
class MyCustomClass:
    def __init__(self, value):
        self.value = value
        self.timestamp = time.time()
    
    def to_dict(self):
        return {"value": self.value, "timestamp": self.timestamp}
    
    @classmethod
    def from_dict(cls, data):
        obj = cls(data["value"])
        obj.timestamp = data["timestamp"]
        return obj

# カスタムシリアライザーを作成
serializer = CustomSerializer()

# シリアライザーを登録
def my_serializer(obj):
    return obj.to_dict()

serializer.register(MyCustomClass, my_serializer)

# カスタムオブジェクトを作成
custom_obj = MyCustomClass(42)
print(f"カスタムオブジェクト: value={custom_obj.value}, timestamp={custom_obj.timestamp}")


カスタムオブジェクト: value=42, timestamp=1766214260.582051


In [21]:
# カスタムシリアライザーを使って保存
save_session("temp/custom_session.pkl", serializer=serializer, verbose=True)


Saved variable: data (dict)
Saved variable: numbers (list)
Saved variable: array (ndarray)
Saved variable: df (DataFrame)
Saved variable: size_normal (int)
Saved variable: size_gzip (int)
Saved variable: size_bz2 (int)
Saved variable: large_data (ndarray)
Saved variable: info (dict)
Saved variable: variables (list)
Saved variable: test_globals (dict)
Saved variable: test_globals2 (dict)
Saved variable: data1 (dict)
Saved variable: data2 (dict)
Saved variable: new_var (str)
Saved variable: result (dict)
Saved variable: var (str)
Saved variable: new_vars (dict)
Saved variable: i (int)
Saved variable: size (int)
Saved variable: serializer (CustomSerializer)
Saved variable: custom_obj (dict)
Session saved to temp/custom_session.pkl (8002656 bytes)
Warnings: 4 variables could not be saved


## 9. 実践例

実際の使用例として、機械学習モデルのセッションを保存・復元する例を示します。


In [22]:
# 機械学習の例（シミュレーション）
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

# データを生成
X, y = make_classification(n_samples=1000, n_features=20, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# モデルを訓練
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# スコアを計算
train_score = model.score(X_train, y_train)
test_score = model.score(X_test, y_test)

print(f"訓練スコア: {train_score:.4f}")
print(f"テストスコア: {test_score:.4f}")

# セッションを保存（モデル、データ、スコアを含む）
save_session("temp/ml_session.pkl", metadata=True, compress=True, verbose=True)


訓練スコア: 1.0000
テストスコア: 0.9000
Saved variable: data (dict)
Saved variable: numbers (list)
Saved variable: array (ndarray)
Saved variable: df (DataFrame)
Saved variable: size_normal (int)
Saved variable: size_gzip (int)
Saved variable: size_bz2 (int)
Saved variable: large_data (ndarray)
Saved variable: info (dict)
Saved variable: variables (list)
Saved variable: test_globals (dict)
Saved variable: test_globals2 (dict)
Saved variable: data1 (dict)
Saved variable: data2 (dict)
Saved variable: new_var (str)
Saved variable: result (dict)
Saved variable: var (str)
Saved variable: new_vars (dict)
Saved variable: i (int)
Saved variable: size (int)
Saved variable: serializer (CustomSerializer)
Saved variable: custom_obj (MyCustomClass)
Saved variable: X (ndarray)
Saved variable: y (ndarray)
Saved variable: X_train (ndarray)
Saved variable: X_test (ndarray)
Saved variable: y_train (ndarray)
Saved variable: y_test (ndarray)
Saved variable: model (RandomForestClassifier)
Saved variable: train_score 

In [23]:
# セッション情報を確認
print_session_info("temp/ml_session.pkl")

Session File: temp/ml_session.pkl
File Size: 8,056,480 bytes
Compression: gzip
Modified: 2025-12-20T16:04:21.744590
Variables: 31
Total Data Size: 17,400,321 bytes

Metadata:
  version: 0.1.1
  saved_at: 2025-12-20T16:04:21.340995
  variable_count: 31

Variables:
  X                    (ndarray        ) - 160,163 bytes
  X_test               (ndarray        ) - 32,153 bytes
  X_train              (ndarray        ) - 128,163 bytes
  array                (ndarray        ) - 188 bytes
  custom_obj           (MyCustomClass  ) - 80 bytes
  data                 (dict           ) - 56 bytes
  data1                (dict           ) - 38 bytes
  data2                (dict           ) - 38 bytes
  df                   (DataFrame      ) - 707 bytes
  i                    (int            ) - 5 bytes
  info                 (dict           ) - 323 bytes
  large_data           (ndarray        ) - 8,000,164 bytes
  model                (RandomForestClassifier) - 1,058,227 bytes
  new_var              

In [24]:
# セッション検証
from SessionSmith import verify_session

is_valid, error = verify_session("temp/ml_session.pkl")
if is_valid:
    print("✓ セッションファイルは有効です")
else:
    print(f"✗ エラー: {error}")


✓ セッションファイルは有効です


## 10. クリーンアップ

チュートリアルで作成したファイルを削除します。


In [25]:
# 作成したファイルを削除
files_to_remove = [
    "temp/my_session.pkl",
    "temp/session_normal.pkl",
    "temp/session_gzip.pkl",
    "temp/session_bz2.pkl",
    "temp/session1.pkl",
    "temp/session2.pkl",
    "temp/managed_session.pkl",
    "temp/autosave.pkl",
    "temp/custom_session.pkl",
    "temp/ml_session.pkl"
]

for file in files_to_remove:
    if os.path.exists(file):
        os.remove(file)
        print(f"✓ 削除: {file}")

# tempディレクトリが空になったら削除
try:
    if os.path.exists("temp") and not os.listdir("temp"):
        os.rmdir("temp")
        print("✓ temp/ディレクトリを削除しました")
except OSError:
    pass

print("\n✓ クリーンアップ完了！")


✓ 削除: temp/my_session.pkl
✓ 削除: temp/session_normal.pkl
✓ 削除: temp/session_gzip.pkl
✓ 削除: temp/session_bz2.pkl
✓ 削除: temp/session1.pkl
✓ 削除: temp/session2.pkl
✓ 削除: temp/managed_session.pkl
✓ 削除: temp/autosave.pkl
✓ 削除: temp/custom_session.pkl
✓ 削除: temp/ml_session.pkl
✓ temp/ディレクトリを削除しました

✓ クリーンアップ完了！


## まとめ

このチュートリアルでは、SessionSmithの主要な機能を学びました：

- ✅ 基本的な保存・復元
- ✅ 圧縮サポート
- ✅ セッション情報表示
- ✅ セレクティブロード
- ✅ セッション比較
- ✅ SessionManagerクラス
- ✅ 自動バックアップ
- ✅ カスタムシリアライザー
- ✅ 実践的な使用例

詳細なドキュメントは [README.md](readme.md) を参照してください。
